In [81]:
import pandas as pd
import numpy as np
import os

In [82]:
pasta_dados = "..\\data\\input"
pasta_dados_saida = "..\\data\\output"

dados_campanha = os.path.join(pasta_dados, 'Campanha Incentivo - Distribuição Vinho.xlsx')
dados_configuracao = os.path.join(pasta_dados, 'Configuracao da Campanha.xlsx')
dados_brutos = os.path.join(pasta_dados, 'Dados_Brutos.csv')

In [83]:
relatorio_campanha_premiacao = pd.read_excel(dados_campanha, sheet_name="Premiação Distribuidores")
relatorio_campanha_detalhamento = pd.read_excel(dados_campanha, sheet_name="Detalhamento Mês Apurado")
relatorio_configuracao_cliente = pd.read_excel(dados_configuracao, sheet_name="Cliente")
relatorio_configuracao_produto = pd.read_excel(dados_configuracao, sheet_name="Produto")
relatorio_bruto = pd.read_csv(dados_brutos, sep=";", encoding="utf-8", dtype={"NUMERODOCUMENTO": "str"})

In [84]:
relatorio_bruto.info()

<class 'pandas.DataFrame'>
RangeIndex: 139741 entries, 0 to 139740
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   CODIGOCLIENTE             139741 non-null  int64
 1   DESCRICAOCLIENTE          139741 non-null  str  
 2   CNPJ                      139741 non-null  str  
 3   CODIGOGRUPOCLIENTE        139741 non-null  int64
 4   DESCRICAOGRUPOCLIENTE     139741 non-null  str  
 5   CODIGOFAMILIA             139741 non-null  int64
 6   DESCRICAOFAMILIA          139741 non-null  str  
 7   CODIGOGRUPOPRODUTO        139741 non-null  int64
 8   DESCRICAOGRUPOPRODUTO     139741 non-null  str  
 9   CODIGOPRODUTO             139741 non-null  int64
 10  NOMEPRODUTO               139741 non-null  str  
 11  CODIGODOCUMENTO           139741 non-null  int64
 12  NUMERODOCUMENTO           139741 non-null  str  
 13  SERIEDOCUMENTO            139741 non-null  int64
 14  DATAEMISSAO               13974

### 1. Filtragem:

In [85]:
CFOPS_CAMPANHA = (5102, 5106, 5110, 6102, 6106, 6110, 5160, 6160, 6910, 5910)

In [86]:
relatorio_apurado = relatorio_bruto[
    (
        relatorio_bruto["CODIGOCLIENTE"].isin(
            relatorio_configuracao_cliente["CODIGOCLIENTE"]
        )
    )
    &
    (
        relatorio_bruto["CODIGOPRODUTO"].isin(
            relatorio_configuracao_produto["CODIGOPRODUTO"]
        )
    )
    &
    (
        relatorio_bruto["CFOP"].isin(CFOPS_CAMPANHA)
    )
]

relatorio_apurado.shape

(566, 22)

### Validação: 
    - Base Bruta x Base Apurada

In [87]:
relatorio_apurado = relatorio_apurado.rename(columns={
    "VOLUME": "VOLUMEBRUTO"
})

relatorio_apurado.columns

Index(['CODIGOCLIENTE', 'DESCRICAOCLIENTE', 'CNPJ', 'CODIGOGRUPOCLIENTE',
       'DESCRICAOGRUPOCLIENTE', 'CODIGOFAMILIA', 'DESCRICAOFAMILIA',
       'CODIGOGRUPOPRODUTO', 'DESCRICAOGRUPOPRODUTO', 'CODIGOPRODUTO',
       'NOMEPRODUTO', 'CODIGODOCUMENTO', 'NUMERODOCUMENTO', 'SERIEDOCUMENTO',
       'DATAEMISSAO', 'CFOP', 'VALORLIQUIDO', 'VALORBRUTOIMPOSTO',
       'VALORVENDALIQUIDO', 'VOLUMEBRUTO', 'TIPOFATURAMENTO',
       'DESCRICAOTIPOFATURAMENTO'],
      dtype='str')

In [88]:
validacao_campanha_dados_apurados = pd.merge(
    relatorio_campanha_detalhamento,
    relatorio_apurado[["CODIGOCLIENTE", "CODIGODOCUMENTO", "CODIGOPRODUTO", "VOLUMEBRUTO"]],
    how="left",
    on=["CODIGODOCUMENTO", "CODIGOCLIENTE", "CODIGOPRODUTO"]
)

In [89]:
validacao_campanha_dados_apurados

,CODIGOCAMPANHA,DESCRICAOCAMPANHA,CODIGOCLIENTE,NOMECLIENTE,CODIGOGRUPOCLIENTE,CODIGODOCUMENTO,NUMERODOCUMENTO,NUMEROSERIE,DATAEMISSAO,CODIGOFAMILIAPRODUTO,...,CODIGOGRUPOPRODUTO,NOMEGRUPOPRODUTO,CODIGOPRODUTO,NOMEPRODUTO,CODIGOCFOP,DESCRICAOTIPOFATURAMENTO,FATORPRODUTO,VOLUME,DATAAPURACAO,VOLUMEBRUTO
0,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729304,17369,1,2026-02-20,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,5102,VENDA,1,50,2026-03-09 13:36:59.000,50.0
1,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5738300,17451,1,2026-02-22,80000007,...,90013032,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,5102,VENDA,1,25,2026-03-09 13:36:59.000,25.0
2,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5738300,17451,1,2026-02-22,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,5102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
3,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5684798,17096,1,2026-02-07,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,5102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
4,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5771351,17686,1,2026-02-07,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5745184,36458,1,2026-02-23,80000007,...,90013032,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
565,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5694289,35822,1,2026-02-10,80000007,...,90013032,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0
566,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5721639,36171,1,2026-02-17,80000007,...,90013032,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,6102,VENDA,1,50,2026-03-09 13:36:58.995,50.0
567,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5745157,36401,1,2026-02-23,80000007,...,90013032,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,6102,VENDA,1,25,2026-03-09 13:36:58.995,25.0


In [90]:
validacao_campanha_dados_apurados["VALIDAÇÃO"] = np.where(
    validacao_campanha_dados_apurados["VOLUME"] != validacao_campanha_dados_apurados["VOLUMEBRUTO"],
    "DIVERGENTE",
    "CORRETO"
)

validacao_campanha_dados_apurados[validacao_campanha_dados_apurados["VALIDAÇÃO"] == "DIVERGENTE"]

,CODIGOCAMPANHA,DESCRICAOCAMPANHA,CODIGOCLIENTE,NOMECLIENTE,CODIGOGRUPOCLIENTE,CODIGODOCUMENTO,NUMERODOCUMENTO,NUMEROSERIE,DATAEMISSAO,CODIGOFAMILIAPRODUTO,...,NOMEGRUPOPRODUTO,CODIGOPRODUTO,NOMEPRODUTO,CODIGOCFOP,DESCRICAOTIPOFATURAMENTO,FATORPRODUTO,VOLUME,DATAAPURACAO,VOLUMEBRUTO,VALIDAÇÃO
14,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729295,17348,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE
15,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,5729295,17348,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE
114,4002,Harmonização Master - Vinho Tintos Importados ...,10004204,LOGÍSTICA VALE DAS BEBIDAS,2,5728490,136345,1,2026-02-20,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,5110,VENDA,1,25,2026-03-09 13:36:58.995,200.0,DIVERGENTE
183,4002,Harmonização Master - Vinho Tintos Importados ...,10004219,SUPRIMENTOS MASTER BEBIDAS,2,5762186,176536,1,2026-02-19,80000007,...,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,1202,DEVOLUÇÃO DE VENDA,1,-10,2026-03-09 13:36:58.995,NaN,DIVERGENTE
346,4002,Harmonização Master - Vinho Tintos Importados ...,10004227,LOGÍSTICA INTEGRAL DE BEBIDAS,2,5744799,44502,1,2026-02-24,80000007,...,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,2202,DEVOLUÇÃO DE VENDA,1,-50,2026-03-09 13:36:58.995,NaN,DIVERGENTE
559,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5703114,67183,1,2026-02-13,80000007,...,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,1202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE
560,4002,Harmonização Master - Vinho Tintos Importados ...,10004228,SUPRIMENTOS BRINDE UNIDO,2,5757643,36677,1,2026-02-28,80000007,...,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,2202,DEVOLUÇÃO DE VENDA,1,-25,2026-03-09 13:36:58.995,NaN,DIVERGENTE


### Validação
    - Premiação da Campanha

In [91]:
def percentual_crescimento(crescimento):
    if crescimento >= 0.2:
        return 0.1

    elif crescimento >= 0.15:
        return 0.08
    
    elif crescimento >= 0.1:
        return 0.06
    
    elif crescimento >= 0.05:
        return 0.03
    
    else:
        return 0
 

In [92]:
campanha_detalhamento = (
    relatorio_campanha_detalhamento.groupby(["CODIGOCLIENTE", "CODIGOPRODUTO"])
    ["VOLUME"]
    .sum()
    .reset_index()
)

campanha_detalhamento.head()

,CODIGOCLIENTE,CODIGOPRODUTO,VOLUME
0,10004006,10001164,200
1,10004006,10003736,50
2,10004006,10024278,100
3,10004201,10001164,1225
4,10004201,10003736,475


In [93]:
validacao_premiacao = pd.merge(
    relatorio_campanha_premiacao,
    campanha_detalhamento,
    how="left",
    on=["CODIGOCLIENTE", "CODIGOPRODUTO"]
)

validacao_premiacao.head()

,CODIGOCAMPANHA,DESCRICAOCAMPANHA,CODIGOCLIENTE,DESCRICAOCLIENTE,CODIGOGRUPOCLIENTE,DESCRICAOGRUPOCLIENTE,FAMILIAPRODUTO,GRUPOPRODUTO,CODIGOPRODUTO,NOMEPRODUTO,METAVOLUMEVENDA,VOLUMEREALIZADO,PORCENTAGEMCRESCIMENTO,PRECOPRODUTO,VALORVENDAPREMIACAO,PERCENTUALPREMIACAO,VALORPREMIACAO,VOLUME
0,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,Distribuidor,Vinho,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,83.3333,50,0.6000,"14,14",707.00,0.0,0.000,50
1,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,Distribuidor,Vinho,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,241.6667,100,0.4138,"12,61",1261.00,0.0,0.000,100
2,4002,Harmonização Master - Vinho Tintos Importados ...,10004006,SUPRIMENTOS GARRAFAS & COPO,2,Distribuidor,Vinho,Vinhos Tintos Importados,10001164,Vinho Tinto Cabernet Reservado 750ml,275.0000,200,0.7273,"21,27",4254.00,0.0,0.000,200
3,4002,Harmonização Master - Vinho Tintos Importados ...,10004201,SUPRIMENTOS PREMIUM BEVERAGES,2,Distribuidor,Vinho,Vinhos Tintos Importados,10003736,Vinho Tinto Malbec Argentino 750ml,333.3333,475,1.4250,"14,14",6716.50,0.1,671.650,475
4,4002,Harmonização Master - Vinho Tintos Importados ...,10004201,SUPRIMENTOS PREMIUM BEVERAGES,2,Distribuidor,Vinho,Vinhos Tintos Importados,10024278,Vinho Tinto Merlot Chileno 750ml,416.6667,725,1.7400,"12,61",9142.25,0.1,914.225,725


In [94]:
validacao_premiacao["CRESCIMENTOCALCULADO"] = (
    (validacao_premiacao["VOLUME"] / validacao_premiacao["METAVOLUMEVENDA"]) - 1
).round(4)

validacao_premiacao["PERCENTUALCALCULADO"] = (
    validacao_premiacao["CRESCIMENTOCALCULADO"].apply(percentual_crescimento)
)

validacao_premiacao["VALORPREMIACAOCALCULADO"] = (
    validacao_premiacao["VALORVENDAPREMIACAO"] * (validacao_premiacao["PERCENTUALCALCULADO"])
).round(4)

validacao_premiacao[["DESCRICAOCLIENTE", "VALORVENDAPREMIACAO", "CRESCIMENTOCALCULADO", "PERCENTUALCALCULADO", "VALORPREMIACAOCALCULADO"]].head()

,DESCRICAOCLIENTE,VALORVENDAPREMIACAO,CRESCIMENTOCALCULADO,PERCENTUALCALCULADO,VALORPREMIACAOCALCULADO
0,SUPRIMENTOS GARRAFAS & COPO,707.00,-0.4000,0.0,0.000
1,SUPRIMENTOS GARRAFAS & COPO,1261.00,-0.5862,0.0,0.000
2,SUPRIMENTOS GARRAFAS & COPO,4254.00,-0.2727,0.0,0.000
3,SUPRIMENTOS PREMIUM BEVERAGES,6716.50,0.4250,0.1,671.650
4,SUPRIMENTOS PREMIUM BEVERAGES,9142.25,0.7400,0.1,914.225


In [95]:
validacao_premiacao["VALIDAÇÃO"] = np.select(
    [
        ~np.isclose(
            validacao_premiacao["PERCENTUALCALCULADO"],
            validacao_premiacao["PERCENTUALPREMIACAO"]
        ),

        ~np.isclose(
            validacao_premiacao["VALORPREMIACAOCALCULADO"],
            validacao_premiacao["VALORPREMIACAO"]
        )
    ],
    [
        "PERCENTUAL DIVERGENTE",
        "PREMIACAO DIVERGENTE",
    ],
    default="CORRETO"
)

validacao_premiacao[(validacao_premiacao["VALIDAÇÃO"] == "PERCENTUAL DIVERGENTE") | (validacao_premiacao["VALIDAÇÃO"] == "PREMIACAO DIVERGENTE")][["VOLUME", "METAVOLUMEVENDA", "CRESCIMENTOCALCULADO", "PERCENTUALPREMIACAO", "PERCENTUALCALCULADO", "VALORPREMIACAO", "VALORPREMIACAOCALCULADO", "VALIDAÇÃO"]]

,VOLUME,METAVOLUMEVENDA,CRESCIMENTOCALCULADO,PERCENTUALPREMIACAO,PERCENTUALCALCULADO,VALORPREMIACAO,VALORPREMIACAOCALCULADO,VALIDAÇÃO
7,175,191.6667,-0.087,0.1,0.0,530.25,0.0,PERCENTUAL DIVERGENTE


In [97]:
with pd.ExcelWriter(os.path.join(pasta_dados_saida, "Indicadores_ValidacaoS.xlsx"), engine="xlsxwriter") as writer:
    validacao_campanha_dados_apurados.to_excel(
        writer,
        sheet_name="Dados Apurados x Dados Brutos",
        index=False
    )

    validacao_premiacao.to_excel(
        writer,
        sheet_name="Premiação",
        index=False
    )